In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import jax
import numpy as np
from jax import numpy as jnp
import netket as nk
import flax.linen as nn

/home/chang/soft/miniconda3/envs/mindquantum_10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nk.config.netket_random_state_fallback_warning = False

In [3]:
n_chains = 10
nqubits = 14
norb = int(nqubits/2)
Na = Nb = 5
alpha=3
discard = 5*10**2
sample_size = 10 ** 3
seed = 0
seed_params = 0
σp = jnp.array([[1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1]]*n_chains,dtype=jnp.int8)
ob_string_list = jnp.load('../../spectral_gap/H2O/string_op/ob_string_list_R_2.0.npy')
coeff_list = jnp.load('../../spectral_gap/H2O/string_op/coeff_list_R_2.0.npy')

In [4]:
σp.shape

(10, 14)

In [5]:
import jax

from netket.utils import struct
from netket.utils.types import Scalar, Array

from netket.hilbert.constraint import DiscreteHilbertConstraint


class ABConstraint(DiscreteHilbertConstraint):
    """
    Constraint of an Hilbert space enforcing the number of alpha and beta electrons in the degrees of freedom.
    """

    Na: Scalar = struct.field(pytree_node=False)
    Nb: Scalar = struct.field(pytree_node=False)

    def __init__(self, Na: Scalar, Nb: Scalar):
        if Na is None or Nb is None:
            raise TypeError("Na and Nb must be a number.")

        self.Na = Na
        self.Nb = Nb

    @jax.jit
    def __call__(self, x: Array) -> Array:
        return jax.numpy.logical_and((x*0.5+0.5)[..., 0::2].sum(axis=-1)== round(self.Na), (x*0.5+0.5)[..., 1::2].sum(axis=-1)== round(self.Nb))

    def __hash__(self):
        return hash(("ABConstraint", self.Na, self.Nb))

    def __eq__(self, other):
        if isinstance(other, ABConstraint):
            return jax.numpy.logical_and(self.Na == other.Na, self.Nb == other.Nb)
        return False

    def __repr__(self):
        return f"ABConstraint({self.Na, self.Nb})"

In [6]:
hilbert = nk.hilbert.Spin(0.5, nqubits, constraint=ABConstraint(Na=Na, Nb=Nb),inverted_ordering=True)
#hilbert = nk.hilbert.Spin(0.5, nqubits,inverted_ordering=True)
operator = nk.operator.PauliStrings(hilbert, ob_string_list, coeff_list)
opdopc_nolocal =  nk.operator.PauliStrings(hilbert,
                                 np.array(['I'*nqubits, 'Z'+'I'*(nqubits-1), 'I'*(nqubits-1)+'Z', 'Z'+'I'*(nqubits-2)+'Z']),
                                 jnp.array([0.25, -0.25, -0.25, 0.25])) # Double occupancy operator

# ExcitationSD rule

In [7]:
import sys
sys.path.append('../../../')
from Rule import ExcitationSDRule

In [8]:
rule = ExcitationSDRule(jnp.zeros(norb), jnp.zeros(Na), jnp.zeros(Nb))
sa = nk.sampler.MetropolisSampler(hilbert, rule, n_chains=n_chains, reset_chains=False, sweep_size=1)                   # construct sampler
model = nk.models.RBMModPhase(alpha=alpha, param_dtype=float, kernel_init=nn.initializers.normal(stddev=0.01))          # RBMModPhase ansatz
sr = nk.optimizer.SR(diag_shift=0.03, diag_scale=0.03,
                    holomorphic=False,  # 如果ansatz是bool类型就写这个True
                    )                                                                                                   # Stochastic Reconfiguration
vs = nk.vqs.MCState(sa, model,n_discard_per_chain=1,n_samples=sample_size)                                              # neural network quantum state
vs.n_discard_per_chain = discard                                                                                        # discard size for each chain
vs.init_parameters(seed=seed_params, init_fun=jax.nn.initializers.normal(stddev=0.01))                                  # Initialize parameters
sample_state_new = vs.sampler_state
sample_state_new.replace(σ=σp)
vs.sampler_state = sample_state_new                                                                                     # Sampler setting
save = 'ExcitationSD/data'
opt = nk.optimizer.Adam(learning_rate=0.002)                                                                            # Adam optimizer
gs = nk.driver.VMC(operator, opt, variational_state=vs, preconditioner=sr)                                              # VMC
gs.run(n_iter=6000, out=save, obs={'opdopc_nolocal': opdopc_nolocal,})

  0%|          | 25/6000 [00:09<38:48,  2.57it/s, Energy=-67.43-0.01j ± 0.44 [σ²=110.29, R̂=1.0243]] 


KeyboardInterrupt: 

# Uniform rule

In [9]:
import sys
sys.path.append('../../../')
from Rule import UniformRule

In [10]:
rule = UniformRule(hilbert.all_states())
sa = nk.sampler.MetropolisSampler(hilbert, rule, n_chains=n_chains, reset_chains=False, sweep_size=1)                   # construct sampler
model = nk.models.RBMModPhase(alpha=alpha, param_dtype=float, kernel_init=nn.initializers.normal(stddev=0.01))          # RBMModPhase ansatz
sr = nk.optimizer.SR(diag_shift=0.03, diag_scale=0.03,
                    holomorphic=False,  # 如果ansatz是bool类型就写这个True
                    )                                                                                                   # Stochastic Reconfiguration
vs = nk.vqs.MCState(sa, model,n_discard_per_chain=1,n_samples=sample_size)                                              # neural network quantum state
vs.n_discard_per_chain = discard                                                                                        # discard size for each chain
vs.init_parameters(seed=seed_params, init_fun=jax.nn.initializers.normal(stddev=0.01))                                  # Initialize parameters
sample_state_new = vs.sampler_state
sample_state_new.replace(σ=σp)
vs.sampler_state = sample_state_new                                                                                     # Sampler setting
save = 'Uniform/data'
opt = nk.optimizer.Adam(learning_rate=0.002)                                                                            # Adam optimizer
gs = nk.driver.VMC(operator, opt, variational_state=vs, preconditioner=sr)                                              # VMC
gs.run(n_iter=6000, out=save, obs={'opdopc_nolocal': opdopc_nolocal,})

100%|██████████| 6000/6000 [33:44<00:00,  2.96it/s, Energy=-74.75780+0.00022j ± 0.00046 [σ²=0.00007, R̂=1.1960]]    


(JsonLog('Uniform/data', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    6.7209789752960205
   	Params: 1.4516830444335938,)

# Effective (hopping) rule

In [11]:
from Rule import UnitaryRule

In [12]:
Q_mat_Effective_hopping = jnp.load('../../spectral_gap/H2O/Q_mat/Q_mat_Effective_hopping.npy')
rule = UnitaryRule(Q_mat_Effective_hopping, hilbert.all_states())
sa = nk.sampler.MetropolisSampler(hilbert, rule, n_chains=n_chains, reset_chains=False, sweep_size=1)                   # construct sampler
model = nk.models.RBMModPhase(alpha=alpha, param_dtype=float, kernel_init=nn.initializers.normal(stddev=0.01))          # RBMModPhase ansatz
sr = nk.optimizer.SR(diag_shift=0.03, diag_scale=0.03,
                    holomorphic=False,  # 如果ansatz是bool类型就写这个True
                    )                                                                                                   # Stochastic Reconfiguration
vs = nk.vqs.MCState(sa, model,n_discard_per_chain=1,n_samples=sample_size)                                              # neural network quantum state
vs.n_discard_per_chain = discard                                                                                        # discard size for each chain
vs.init_parameters(seed=seed_params, init_fun=jax.nn.initializers.normal(stddev=0.01))                                  # Initialize parameters
sample_state_new = vs.sampler_state
sample_state_new.replace(σ=σp)
vs.sampler_state = sample_state_new                                                                                     # Sampler setting
save = 'Effective_hopping/data'
opt = nk.optimizer.Adam(learning_rate=0.002)                                                                            # Adam optimizer
gs = nk.driver.VMC(operator, opt, variational_state=vs, preconditioner=sr)                                              # VMC
gs.run(n_iter=6000, out=save, obs={'opdopc_nolocal': opdopc_nolocal,})

100%|██████████| 6000/6000 [40:43<00:00,  2.46it/s, Energy=-74.758079+0.000135j ± 0.000090 [σ²=0.000003, R̂=1.1249]]


(JsonLog('Effective_hopping/data', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    7.5987067222595215
   	Params: 1.3850233554840088,)

# Quantum (hopping) rule

In [13]:
Q_mat_Quantum_hopping = jnp.load('../../spectral_gap/H2O/Q_mat/Q_mat_Quantum_hopping.npy')
rule = UnitaryRule(Q_mat_Quantum_hopping, hilbert.all_states())
sa = nk.sampler.MetropolisSampler(hilbert, rule, n_chains=n_chains, reset_chains=False, sweep_size=1)                   # construct sampler
model = nk.models.RBMModPhase(alpha=alpha, param_dtype=float, kernel_init=nn.initializers.normal(stddev=0.01))          # RBMModPhase ansatz
sr = nk.optimizer.SR(diag_shift=0.03, diag_scale=0.03,
                    holomorphic=False,  # 如果ansatz是bool类型就写这个True
                    )                                                                                                   # Stochastic Reconfiguration
vs = nk.vqs.MCState(sa, model,n_discard_per_chain=1,n_samples=sample_size)                                              # neural network quantum state
vs.n_discard_per_chain = discard                                                                                        # discard size for each chain
vs.init_parameters(seed=seed_params, init_fun=jax.nn.initializers.normal(stddev=0.01))                                  # Initialize parameters
sample_state_new = vs.sampler_state
sample_state_new.replace(σ=σp)
vs.sampler_state = sample_state_new                                                                                     # Sampler setting
save = 'Quantum_hopping/data'
opt = nk.optimizer.Adam(learning_rate=0.002)                                                                            # Adam optimizer
gs = nk.driver.VMC(operator, opt, variational_state=vs, preconditioner=sr)                                              # VMC
gs.run(n_iter=6000, out=save, obs={'opdopc_nolocal': opdopc_nolocal,})

  1%|          | 62/6000 [00:23<38:39,  2.56it/s, Energy=-73.36+0.01j ± 0.17 [σ²=11.41, R̂=1.0641]] 

100%|██████████| 6000/6000 [40:46<00:00,  2.45it/s, Energy=-74.75450+0.00003j ± 0.00031 [σ²=0.00004, R̂=1.0798]]    


(JsonLog('Quantum_hopping/data', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    7.625245571136475
   	Params: 1.400991678237915,)

# Exact

In [14]:
model = nk.models.RBMModPhase(alpha=alpha, param_dtype=float, kernel_init=nn.initializers.normal(stddev=0.01))          # RBMModPhase ansatz
sr = nk.optimizer.SR(diag_shift=0.03, diag_scale=0.03,
                    holomorphic=False,  # 如果ansatz是bool类型就写这个True
                    )       
vs = nk.vqs.FullSumState(hilbert, model)
vs.init_parameters(seed=seed_params, init_fun=jax.nn.initializers.normal(stddev=0.01))
save = 'Exact/data'
opt = nk.optimizer.Adam(learning_rate=0.002)                                                                            # Adam optimizer
gs = nk.driver.VMC(operator, opt, variational_state=vs, preconditioner=sr)                                              # VMC
gs.run(n_iter=6000, out=save, obs={'opdopc_nolocal': opdopc_nolocal,})

100%|██████████| 6000/6000 [03:32<00:00, 28.26it/s, Energy=-7.476e+01-2.198e-15j ± 0.000e+00 [σ²=3.917e-04]]


(JsonLog('Exact/data', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    0.9757881164550781
   	Params: 0.852736234664917,)